Import libraries

In [14]:
pip install pymoo

Note: you may need to restart the kernel to use updated packages.


In [15]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
from matplotlib import pyplot as plt
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import ElementwiseProblem
from pymoo.optimize import minimize
from pymoo.core.variable import Real, Binary
from pymoo.core.mixed import MixedVariableSampling, MixedVariableMating, MixedVariableDuplicateElimination

In [16]:
#suppress warning
import warnings

warnings.filterwarnings("ignore")

Load models

In [17]:
H2_model = joblib.load("H2_model")
CO2_model = joblib.load("CO2_model")
CO_model = joblib.load("CO_model")
Tar_model = joblib.load("Tar_model")

Import Data

In [18]:
#Import Original data
org_predictor = pd.read_excel("org_predictor.xlsx").drop("Unnamed: 0", axis=1)
org_target = pd.read_excel("Clean_target.xlsx").drop("Unnamed: 0", axis=1)
org_data = pd.concat([org_predictor, org_target], axis=1)

X_data = pd.read_excel("Treated_predictor.xlsx")

In [19]:
# define x boundary
xl = pd.DataFrame.min(X_data)
xu = pd.DataFrame.max(X_data)

xl_real = []
xu_real = []
for i in range(11):
    xl_real.append(xl[i])
    xu_real.append(xu[i])

In [20]:
# define denormalisation function
def denormalize(norm_arr, org_arr, u, l):
    denorm_arr = []
    diff = u - l
    diff_arr = np.nanmax(org_arr) - np.nanmin(org_arr)
    for xnorm in norm_arr:
        real = (((xnorm - l) * diff_arr) / diff) + np.nanmin(org_arr)
        denorm_arr.append(real)
    return denorm_arr


def normalize(org_arr):
    norm = []
    # xnorm = (x-xmin)/(xmax-xmin)
    for i in range(11):
        xnorm = (org_arr[0][i] - xl_real[i]) / (xu_real[i] - xl_real[i])
        norm.append(xnorm)
    for i in range(11, 34):
        norm.append(org_arr[0][i])
    return [norm]


def rescale(x, var, feed_type=None):
    varmin = np.min(X_data[var])
    varmax = np.max(X_data[var])
    l = np.min(X_data[X_data[feed_type] == 1][var])
    u = np.max(X_data[X_data[feed_type] == 1][var])
    return l + (x - varmin) * (u - l) / (varmax - varmin)

In [21]:
# product column names
pred_col = ["N2", "H2", "CO", "CO2", "CH4", "C2Hn", "gas_LHV", "gas_tar", "gas_yield", "char_yield"]

Define problem class

In [22]:
class MyProblem(ElementwiseProblem):

    # define types of x and number of objectives and constraints
    def __init__(self, **kwargs):

        variables = dict()

        # continuous
        for i in range(11):
            variables[X_data.columns[i]] = Real(bounds=(xl_real[i], xu_real[i]))

        # categorical
        for i in range(11, 34):
            variables[X_data.columns[i]] = Binary()

        super().__init__(vars=variables,
                         n_obj=4,
                         n_ieq_constr=17,
                         n_eq_constr=9,
                         **kwargs)

    def _evaluate(self, x, out, *args, **kwargs):

        x = np.array([[x[X_data.columns[i]] for i in range(34)]])

        g20 = g21 = np.ones(1)
        for i in range(19, 24):

            if x[:, i] == 1 and np.sum(x[:, 19:24]) == 1:
                if i == 19 and (x[:, 18] == 1 or (x[:, 14] !=1 and x[:, 15] != 1 and x[:, 16] != 1 and x[:, 17] != 1 and x[:,18] != 1)):  # chips
                    Sizemin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    Sizemax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    g20 = Sizemin - x[:, 0]
                    g21 = x[:, 0] - Sizemax

                elif i == 20 and x[:, 18] == 1:  # dust
                    Sizemin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    Sizemax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    g20 = Sizemin - x[:, 0]
                    g21 = x[:, 0] - Sizemax

                elif i == 21 and (x[:, 14] == 1 or x[:, 18] == 1):  # fibres
                    Sizemin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    Sizemax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    g20 = Sizemin - x[:, 0]
                    g21 = x[:, 0] - Sizemax

                elif i == 22 and (x[:, 14] == 1 or x[:, 15] == 1 or x[:, 17] == 1 or x[:,18] == 1):  # particles
                    Sizemin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    Sizemax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    g20 = Sizemin - x[:, 0]
                    g21 = x[:, 0] - Sizemax

                elif i == 23 and x[:, 17] != 1:  # pellets
                    Sizemin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    Sizemax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Particle size (mm)"])
                    g20 = Sizemin - x[:, 0]
                    g21 = x[:, 0] - Sizemax

                else:
                    for j in range(19, 24):
                        x[:, j] = 0

        if x[:, 30] == 1 or x[:, 31] == 1:
            h8 = x[:, 14]
        else:
            h8 = np.zeros(1)

        # normalise
        xnorm = np.array(normalize(x))

        y_pred = H2_model.predict(xnorm)
        df_pred = pd.DataFrame(y_pred, columns=["H2 (% db)"])
        H2 = df_pred["H2 (% db)"].values

        H2 = -np.array(denormalize(H2, org_data["H2 (% db)"], 1, 0))  # max H2

        y_pred = CO2_model.predict(xnorm)
        df_pred = pd.DataFrame(y_pred, columns=["CO2 (% db)"])
        CO2 = df_pred["CO2 (% db)"].values

        CO2 = np.array(denormalize(CO2, org_data["CO2 (% db)"], 1, 0))  # min CO2
        
        y_pred = CO_model.predict(xnorm)
        df_pred = pd.DataFrame(y_pred, columns=["CO (% db)"])
        CO = df_pred["CO (% db)"].values

        CO = np.array(denormalize(CO, org_data["CO (% db)"], 1, 0))  # min CO
        
        y_pred = Tar_model.predict(xnorm)
        df_pred = pd.DataFrame(y_pred, columns=["Tar (g/Nm3)"])
        Tar = df_pred["Tar (g/Nm3)"].values

        Tar = np.array(denormalize(Tar, org_data["Tar (g/Nm3)"], 1, 0))  # min CO2

        # inequality constraint (g <= 0)
        g1 = x[:, 12] - 1  # catalyst use

        # equality constraints (h = 0)
        h1 = np.sum(x[:, 14:19], axis=1) - 1  # feedstock type
        h2 = np.sum(x[:, 19:24], axis=1) - 1  # feedstock shape
        h3 = np.sum(x[:, 11:12], axis=1) - 1  # operation mode
        h4 = np.sum(x[:, 24:29], axis=1) - 1  # gasifying agent
        h5 = np.sum(x[:, 29:31], axis=1) - 1  # reactor type
        h6 = np.sum(x[:, 31:34], axis=1) - 1  # bed material
        h7 = np.sum(x[:, 13:14], axis=1) - 1  # system scale

        for i in range(14, 19):

            if x[:, i] == 1:
                Cmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["C (%daf)"])
                Cmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["C (%daf)"])
                g2 = Cmin - x[:, 2]
                g3 = x[:, 2] - Cmax

                Hmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["H (%daf)"])
                Hmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["H (%daf)"])
                g4 = Hmin - x[:, 3]
                g5 = x[:, 3] - Hmax
                
                Omin = np.min(X_data[X_data[X_data.columns[i]] == 1]["O (%daf)"])
                Omax = np.max(X_data[X_data[X_data.columns[i]] == 1]["O (%daf)"])
                g6 = Omin - x[:, 4]
                g7 = x[:, 4] - Omax

                Nmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["N (%daf)"])
                Nmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["N (%daf)"])
                g8 = Nmin - x[:, 5]
                g9 = x[:, 5] - Nmax

                Smin = np.min(X_data[X_data[X_data.columns[i]] == 1]["S (%daf)"])
                Smax = np.max(X_data[X_data[X_data.columns[i]] == 1]["S (%daf)"])
                g10 = Smin - x[:, 6]
                g11 = x[:, 6] - Smax

                Ashmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Ash (%db)"])
                Ashmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Ash (%db)"])
                g12 = Ashmin - x[:, 7]
                g13 = x[:, 7] - Ashmax

                Mmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["Moisture (%wb)"])
                Mmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["Moisture (%wb)"])
                g14 = Mmin - x[:, 8]
                g15 = x[:, 8] - Mmax

                #VMmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["feed_VM"])
                #VMmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["feed_VM"])
                #g16 = VMmin - x[:, 9]
                #g17 = x[:, 9] - VMmax

                #FCmin = np.min(X_data[X_data[X_data.columns[i]] == 1]["feed_FC"])
                #FCmax = np.max(X_data[X_data[X_data.columns[i]] == 1]["feed_FC"])
                #g18 = FCmin - x[:, 10]
                #g19 = x[:, 10] - FCmax

            else:
                g2 = np.ones(1)
                g3 = np.ones(1)
                g4 = np.ones(1)
                g5 = np.ones(1)
                g6 = np.ones(1)
                g7 = np.ones(1)
                g8 = np.ones(1)
                g9 = np.ones(1)
                g10 = np.ones(1)
                g11 = np.ones(1)
                g12 = np.ones(1)
                g13 = np.ones(1)
                g14 = np.ones(1)
                g15 = np.ones(1)
                #g16 = np.ones(1)
                #g17 = np.ones(1)
                #g18 = np.ones(1)
                #g19 = np.ones(1)

        #g22 = np.sum(x[:, 1:6], axis=1) - 100.5  # ultimate analysis
        #g23 = 99.5 - np.sum(x[:, 1:6], axis=1)
        #g24 = np.sum(x[:, 7:11], axis=1) - 100.5  # proximate analysis
        #g25 = 99.5 - np.sum(x[:, 7:11], axis=1)
        h9 = np.sum(x[:, 1:6], axis=1) - 100

        out["F"] = [H2, CO2, CO2, Tar]
        out["G"] = [g1, g2, g3, g4, g5, g6, g7, g8, g9, g10,
                    g11, g12, g13, g14, g15, g20, g21]
        # print(out["G"])
        out["H"] = [h1, h2, h3, h4, h5, h6, h7, h8, h9]

In [23]:
problem = MyProblem()

In [24]:
# define algorithm
algorithm = NSGA2(pop_size=100, crossover=0.8, mutation=0.2,
                  sampling=MixedVariableSampling(),
                  mating=MixedVariableMating(eliminate_duplicates=MixedVariableDuplicateElimination()),
                  eliminate_duplicates=MixedVariableDuplicateElimination()
                  )

In [ ]:
# finding results
res = minimize(problem,
               algorithm,
               ("n_gen", 100)
               )

if res is None:
    print("Optimization did not converge.")
else:
    print("Optimization is successful.")
    print(res.X)
    print(res.F)
    print(res.H)
    print(res.G)

# BEEP BEEP!
import winsound

duration = 500
freq = 500
winsound.Beep(freq, duration)
winsound.Beep(freq, duration)

X = res.X

X_list = []
for i in range(len(X)):
    X_list.append(np.array([X[i][X_data.columns[j]] for j in range(39)]))

F = res.F
G = res.G
H = res.H

col = [*X_data.columns, *["H2 (% db)", "CO2 (% db)", "CO (% db)", "Tar (g/Nm3)"]]

rows = []
for i in range(len(F)):
    rows.append(np.concatenate((X_list[i], -F[i][0], F[i][1], F[i][2], F[i][3]), axis=None))

df = pd.DataFrame(rows, columns=col)
print(df)
df.to_excel('MOO_dumb.xlsx', index=False)